# 第16章　利率期权（Cap/Floor/Swaption）

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch16_ir_options.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch16_ir_options.ipynb)

复现例16.1-16.4（caplet/Cap/Floor/Swaption/隐含波动率）、Cap-Floor 平价、图16-1，并与 QuantLib 对拍。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import rateopt as ro
from fi import plotting
plotting.use_chinese_style()
resets, taus = [1, 2, 3, 4], [1, 1, 1, 1]
pay_dfs = [1.03 ** -t for t in (2, 3, 4, 5)]
fwds = [0.03] * 4; N = 1e8; vol = 0.20


## 例16.1　单个 caplet（重置 1y、支付 2y）


In [ ]:
cl = ro.black_caplet(0.03, 0.03, vol, 1, 1, pay_dfs[0], N, 'cap')
print(f'caplet 价值 = {cl:,.0f} 元')


## 例16.2　ATM Cap / Floor 与 Cap-Floor 平价


In [ ]:
for K in (0.02, 0.03, 0.04):
    cap = ro.black_cap(fwds, K, vol, resets, taus, pay_dfs, N, 'cap')
    flr = ro.black_cap(fwds, K, vol, resets, taus, pay_dfs, N, 'floor')
    swp = N * (sum(fwds[i]*taus[i]*pay_dfs[i] for i in range(4)) - K*sum(pay_dfs))
    print(f'K={K*100:.0f}%: Cap={cap:>12,.0f}  Floor={flr:>12,.0f}  Cap-Floor={cap-flr:>12,.0f}  payerSwap={swp:>12,.0f}')


## 图16-1　Cap/Floor 价值随执行价（编程实验 7）


In [ ]:
strikes = np.linspace(0.015, 0.045, 31)
caps = [ro.black_cap(fwds, K, vol, resets, taus, pay_dfs, N, 'cap')/1e4 for K in strikes]
floors = [ro.black_cap(fwds, K, vol, resets, taus, pay_dfs, N, 'floor')/1e4 for K in strikes]
fig, ax = plotting.new_axes()
ax.plot(strikes*100, caps, label='Cap 价值'); ax.plot(strikes*100, floors, label='Floor 价值')
ax.axvline(3.0, ls=':', color='gray', label='ATM (3%)')
ax.set_xlabel('执行利率 K (%)'); ax.set_ylabel('价值（万元）')
ax.set_title('图16-1　Cap 与 Floor 价值随执行价'); ax.legend()
fig.tight_layout()


## 例16.3　Swaption（1y -> 4y payer）


In [ ]:
sp = ro.black_swaption(0.03, 0.03, vol, expiry=1, swap_annuity=sum(pay_dfs), notional=N, kind='payer')
print(f'payer swaption 价值 = {sp:,.0f} 元')


## 例16.4　隐含波动率 + 波动率微笑（编程实验 8）


In [ ]:
iv = ro.implied_vol(cl, 0.03, 0.03, 1, 1, pay_dfs[0], N, 'cap')
print(f'由 caplet 价格反求隐含波动率 = {iv*100:.2f}%（应=20%）')

# 合成一个带偏斜的波动率，造价格，再反求 -> 画微笑
strikes2 = np.linspace(0.02, 0.04, 9)
true_vol = lambda K: 0.20 + 4.0*(K-0.03)**2/0.0001*0.002   # 人造微笑
prices = [ro.black_caplet(0.03, K, true_vol(K), 1, 1, pay_dfs[0], N, 'cap') for K in strikes2]
ivs = [ro.implied_vol(p, 0.03, K, 1, 1, pay_dfs[0], N, 'cap') for p, K in zip(prices, strikes2)]
fig, ax = plotting.new_axes()
ax.plot(strikes2*100, np.array(ivs)*100, marker='o')
ax.set_xlabel('执行利率 K (%)'); ax.set_ylabel('隐含波动率 (%)')
ax.set_title('波动率微笑（合成示例）')
fig.tight_layout()


## 16.7　QuantLib Cap 对拍（编程实验 9）


In [ ]:
import QuantLib as ql
today = ql.Date(15, 6, 2026); ql.Settings.instance().evaluationDate = today
dc, cal = ql.Actual365Fixed(), ql.NullCalendar()
ts = ql.YieldTermStructureHandle(ql.FlatForward(today, 0.03, dc))
idx = ql.IborIndex('Idx', ql.Period(1, ql.Years), 0, ql.CNYCurrency(), cal,
                   ql.Unadjusted, False, dc, ts)
sched = ql.Schedule(today, today + ql.Period(5, ql.Years), ql.Period(1, ql.Years), cal,
                    ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Forward, False)
cap = ql.Cap(ql.IborLeg([1e8], sched, idx), [0.03])
cap.setPricingEngine(ql.BlackCapFloorEngine(ts, ql.QuoteHandle(ql.SimpleQuote(0.20))))
fi_cap = ro.black_cap(fwds, 0.03, vol, resets, taus, pay_dfs, N, 'cap')
print(f'fi.rateopt ATM Cap = {fi_cap:,.0f} 元')
print(f'QuantLib   ATM Cap = {cap.NPV():,.0f} 元（差异来自计息惯例/远期与折现精度）')


---

> 小结：利率期权用 Black 模型定价，caplet 到期=重置日、折现用支付日；Cap-Floor=payer 互换；
> 隐含波动率与波动率曲面是交易核心；可赎回债内嵌 receiver swaption，至此期权这条线打通。
